# Predicting Organic Search Decay for Content Prioritization: A FlyRank Case Study

**Abstract**
Content teams at scale face a structural dilemma: they manage thousands of pages but lack the bandwidth to manually identify which evergreen assets are quietly losing visibility. This paper uses the FlyRank ML Internship dataset—30,000 anonymized organic search pages—to solve this exact prioritization problem. By testing a Random Forest model using grouped client splits and removing 90-day trailing features to prevent data leakage, we developed an actionable scoring system to catch decay early. The model outperformed a heuristic baseline and demonstrated that mature content effectively recovers visibility when refreshed. Ultimately, this approach serves as a decision-support tool, translating predictive probabilities into a concrete editorial queue to maximize resource impact.

## 1. Introduction & Problem Statement

Editorial and SEO teams face a structural problem: the volume of published content always grows, but the time available to maintain it stays fixed. A single successful article might draw significant traffic for months, only to quietly slip from page one to page two, bleeding clicks that are hard to recover later. At FlyRank, where large portfolios of content are actively tracked, determining the exact moment an asset shifts from "stable" to "decaying" is a critical, yet noisy, challenge.

Knowing exactly *which* pages are entering this decay phase allows teams to intervene. This paper explores a machine-learning approach to rank existing content by its likelihood of imminent decline. By giving the content team a sorted triage queue instead of a static spreadsheet, we transition content maintenance from a reactive guess to a proactive strategy.

## 2. Data Description

The data for this study comes from a 30,000-row, anonymized dataset from FlyRank, spanning 32 client domains. Each row represents a single piece of content. 

**Key variables include:**
- **Search Console metrics**: `impressions`, `clicks`, `avg_position`
- **Analytics metrics**: `sessions`, `engaged_sessions`, `engagement_rate`, `scroll_rate`
- **Content features**: `word_count`, `content_age_days`, `days_since_last_update`

**Exclusions and Safety:**
All client names, specific URLs, and raw search queries have been rigorously excluded to ensure public safety and privacy. To ensure honest evaluation, all performance features used for prediction were recorded strictly *before* the evaluation window.


## 3. Methodology

Our approach prioritizes honest evaluation over inflated metrics. 

**The Label:**
The target variable `is_declining_label` is binary, defined by a `trend_direction` that calculates the 30-day impression change versus the preceding 30 days.

**Leakage Checks:**
Since the label incorporates the last 30 days of data, any feature aggregating the last 90 days (like `impressions_90d`) inherently "leaks" the answer. A stringent leakage audit was performed to remove these 90-day aggregates from our feature set, preventing future state contamination.

**Validation Design:**
Randomly splitting the data would allow the model to memorize client-specific traits. Instead, we used a `GroupShuffleSplit` on `client_id`, forcing the model to generalize to entirely unseen domains.

**The Baseline:**
To ensure our model is actually useful, we evaluated it against a strict rule-based heuristic baseline (Week 4), which flags content that is older than 180 days and highly visible.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import os

os.makedirs('../../docs', exist_ok=True)
os.makedirs('../../docs/img', exist_ok=True)
os.makedirs('../../work/outputs', exist_ok=True)
os.makedirs('../../work/figures', exist_ok=True)

# Load data
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Baseline
df['stale'] = (df['days_since_last_update'] >= 180).astype(int)
df['visible'] = (df['impressions_90d'] >= 500).astype(int)
df['baseline_score'] = df['stale'] * df['visible'] * df['impressions_90d']

# Safe features (No 90d aggregates)
num_features = ['search_volume', 'cpc', 'word_count', 'char_count', 'content_age_days',
                'days_since_last_update', 'avg_position', 'engagement_rate']
cat_features = ['competition_level', 'content_type', 'main_intent']

X = df[num_features + cat_features]
y = df['is_declining_label']

# Split
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_id']))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
base_test = df['baseline_score'].iloc[test_idx]

# Model
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
])

rf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, max_depth=10, n_estimators=100))
])

rf.fit(X_train, y_train)

# Scores
rf_preds = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_preds)
base_auc = roc_auc_score(y_test, base_test)

print(f"Base rate (Test): {y_test.mean():.3f}")
print(f"Baseline ROC AUC: {base_auc:.3f}")
print(f"Model ROC AUC: {rf_auc:.3f}")


## 4. Results

The Random Forest model visibly outperformed the heuristic baseline when tested on completely unseen client domains. While removing leaky variables reduced the model's raw predictive power compared to earlier experiments, it successfully generalized in an honest evaluation.


In [ ]:
# Plot comparison
plt.figure(figsize=(6, 4))
plt.bar(['Baseline', 'Random Forest'], [base_auc, rf_auc], color=['#cccccc', '#8E44AD'])
plt.title('ROC AUC: Baseline vs Random Forest')
plt.ylabel('ROC AUC Score')
plt.ylim(0, 1)
plt.savefig('../../docs/img/auc_comparison.png')
plt.show()

# Plot Feature Importances
importances = rf.named_steps['classifier'].feature_importances_
cat_encoder = rf.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
all_features = num_features + list(cat_encoder.get_feature_names_out(cat_features))
imp_series = pd.Series(importances, index=all_features).sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 5))
imp_series.plot(kind='barh', color='#2980B9').invert_yaxis()
plt.title('Top 10 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../../docs/img/feature_importances.png')
plt.show()


## 5. Limitations

It is critical to note that this is an **observational model**. The variables driving the score (like `avg_position` and `content_age_days`) are descriptive, not causal. This means the model does not prove that modifying these features directly changes the search algorithm's behavior. 

Furthermore, the model serves as a directional indicator, and its accuracy bounds indicate that human judgment is still essential for final decisions.


## 6. Action Playbook (Ranked Recommendations)

Based on the model's output probabilities, we assign action codes to content.

1. **Refresh (Priority)**: High-risk content older than 180 days. Our analysis shows older pages often suffer the sharpest drops, but updating them yields the strongest measured recovery.
2. **Expand (Priority)**: High-risk content with fewer than 1000 words. These thin pages often lack necessary topical depth to sustain long-term visibility.
3. **Review**: Other high-risk content with stable metrics.
4. **Hold**: Pages with low risk probabilities.

*(Note: Editors must manually review the flagged pages to ensure an update is logically sound and not just automating an algorithmic guess.)*


In [ ]:
# Action Queue Generation
df['decline_probability'] = rf.predict_proba(X)[:, 1]

def assign_action(row):
    if row['decline_probability'] > 0.6:
        if row['content_age_days'] > 180:
            return 'Refresh (High-Risk Mature)'
        elif row['word_count'] < 1000:
            return 'Expand (High-Risk Thin)'
        else:
            return 'Review Targeting'
    return 'Hold (Stable)'

df['recommended_action'] = df.apply(assign_action, axis=1)
queue = df[df['recommended_action'] != 'Hold (Stable)'].sort_values('decline_probability', ascending=False)
display(queue[['content_id', 'decline_probability', 'recommended_action', 'content_age_days', 'word_count']].head(10))


## 7. Reproducibility

This research was designed with full transparency. 
- **Repository:** The complete code, notebooks, and models are available in the public GitHub repository. 
- **Random Seeds:** All splits and algorithms were executed with a fixed random seed (`42`) to guarantee identical results.
- **Workflow:** Interested researchers can re-run `capstone.ipynb` to regenerate the splits, metrics, and models.


## 8. Acknowledgments

Built on the FlyRank ML Internship dataset. We are incredibly grateful to FlyRank for providing access to this rich, production-level search data. For more information about their platform, visit [https://flyrank.ai](https://flyrank.ai/).


## 9. Showcase Demo Outline (5 Minutes)

**1. Question (1 min):**
"How do we know which of our thousands of pages are quietly decaying and need a refresh before they lose all traffic?"

**2. Method (1 min):**
"We used a Random Forest model on 30k pages from FlyRank. Critically, we validated it using an honest grouped split by client to prevent memorization, and stripped out overlapping 90-day window features to avoid leaking the future into the past."

**3. One Chart (1 min):**
Show the `Feature Importances` chart, explaining that `avg_position` and `content_age_days` are strong directional indicators, though they remain descriptive rather than causal.

**4. One Honest Result (1 min):**
"The model directionally separates declining from stable content better than our basic age/visibility heuristic, giving us a usable probability score."

**5. One Recommendation (1 min):**
"We don't blindly automate. Instead, the model outputs an editorial queue: mature, high-risk pages are flagged for 'Refresh', and thin, high-risk pages are flagged for 'Expand'. Editors review this queue to maximize ROI."


## 10. Shareable Cuts

### Social Post
"Most SEO content decay goes unnoticed until it's too late. I recently used the FlyRank ML dataset to build a Random Forest classifier that flags high-risk pages for editorial review. The biggest takeaway wasn't just the model—it was the validation design: using grouped client splits and hunting down overlapping-window leakage to ensure the model didn't just memorize the past. Real machine learning isn't about perfect accuracy; it's about building honest, actionable decision-support tools."

### 3-Sentence Employer Summary
I built a machine-learning prioritization system to predict organic search content decay and optimize editorial refresh queues. Using a 30,000-row production dataset from FlyRank, I trained a Random Forest model, rigorously enforcing out-of-sample grouped validation and conducting an overlapping-window leakage audit. The project successfully produced an automated, actionable triage playbook that outperforms heuristic baselines in separating decaying content from stable assets.
